# Olist E-Commerce - 02. Sales & Product Analysis
## In-Depth Commercial Analysis, Seasonality & Product Hierarchy

### Objective
This notebook investigates revenue trajectories, sales seasonality, product category margins, freight economics, and Average Order Value (AOV) across the Brazilian marketplace.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path('..').resolve()
DATA_CLEAN = PROJECT_ROOT / 'data' / 'processed' / 'cleaned'

orders = pd.read_csv(DATA_CLEAN / 'olist_orders_dataset.csv', parse_dates=['order_purchase_timestamp'])
order_items = pd.read_csv(DATA_CLEAN / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_CLEAN / 'olist_products_dataset.csv')
translation = pd.read_csv(DATA_CLEAN / 'product_category_name_translation.csv')

print("Orders:", len(orders), "| Order Items:", len(order_items), "| Products:", len(products))


### 1. Build Merged Commercial Fact Table


In [ ]:
# Merge products with English category translation
products_en = products.merge(translation, on='product_category_name', how='left')
products_en['category'] = products_en['product_category_name_english'].fillna('other')

# Merge items with order dates and product categories
df_sales = order_items.merge(orders[['order_id', 'order_status', 'order_purchase_timestamp']], on='order_id', how='inner')
df_sales = df_sales.merge(products_en[['product_id', 'category']], on='product_id', how='left')

df_sales['total_revenue'] = df_sales['price'] + df_sales['freight_value']
df_sales['year_month'] = df_sales['order_purchase_timestamp'].dt.to_period('M')

gross_rev = df_sales['total_revenue'].sum()
prod_sales = df_sales['price'].sum()
freight_val = df_sales['freight_value'].sum()

print("Total Commercial Fact Records:", len(df_sales))
print(f"Gross Revenue: R$ {gross_rev:,.2f}")
print(f"Product Sales: R$ {prod_sales:,.2f} ({(prod_sales/gross_rev)*100:.1f}%)")
print(f"Freight Fees: R$ {freight_val:,.2f} ({(freight_val/gross_rev)*100:.1f}%)")


### 2. Monthly Revenue & Order Volume Trends
Notice the sharp expansion from 2016 through 2018 with key peaks in November 2017 (Black Friday) and May 2018 (Mother's Day).


In [ ]:
monthly_sales = df_sales.groupby('year_month').agg(
    Gross_Revenue=('total_revenue', 'sum'),
    Product_Sales=('price', 'sum'),
    Freight=('freight_value', 'sum'),
    Orders=('order_id', 'nunique'),
    Items_Sold=('order_item_id', 'count')
).reset_index()

monthly_sales['year_month_str'] = monthly_sales['year_month'].astype(str)
monthly_sales['AOV'] = monthly_sales['Gross_Revenue'] / monthly_sales['Orders']

fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(monthly_sales['year_month_str'], monthly_sales['Gross_Revenue'] / 1e6, marker='o', color='#0084FF', linewidth=2.5, label='Gross Revenue (R$ M)')
ax1.plot(monthly_sales['year_month_str'], monthly_sales['Product_Sales'] / 1e6, marker='s', color='#00C49F', linewidth=2, linestyle='--', label='Product Sales (R$ M)')
ax1.plot(monthly_sales['year_month_str'], monthly_sales['Freight'] / 1e6, marker='^', color='#FF8A00', linewidth=2, linestyle=':', label='Freight (R$ M)')
ax1.set_ylabel('Revenue (R$ Millions)', fontsize=12, fontweight='bold')
ax1.set_title('Monthly Revenue Progression (2016 - 2018)', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.bar(monthly_sales['year_month_str'], monthly_sales['Orders'], alpha=0.15, color='#7952B3', label='Order Volume')
ax2.set_ylabel('Total Orders', fontsize=12, fontweight='bold')
ax2.grid(False)

plt.tight_layout()
plt.show()


### 3. Product Category Pareto Analysis (Top 10 Categories)
Evaluating the top merchandise drivers on the platform.


In [ ]:
cat_perf = df_sales.groupby('category').agg(
    Revenue=('total_revenue', 'sum'),
    Product_Sales=('price', 'sum'),
    Freight=('freight_value', 'sum'),
    Orders=('order_id', 'nunique'),
    Units_Sold=('order_item_id', 'count')
).reset_index()

cat_perf = cat_perf.sort_values('Revenue', ascending=False).reset_index(drop=True)
cat_perf['Cumulative_Revenue_%'] = (cat_perf['Revenue'].cumsum() / cat_perf['Revenue'].sum() * 100).round(2)
cat_perf['Avg_Price'] = (cat_perf['Product_Sales'] / cat_perf['Units_Sold']).round(2)

print("Top 10 Categories by Revenue:")
display(cat_perf.head(10))

# Plot top 10 categories
fig, ax = plt.subplots(figsize=(12, 6))
top10 = cat_perf.head(10)
bars = ax.barh(top10['category'][::-1], top10['Revenue'][::-1] / 1e6, color='#0084FF')
ax.set_title('Top 10 Product Categories by Gross Revenue (R$ Millions)', fontsize=14, fontweight='bold')
ax.set_xlabel('Gross Revenue (R$ Millions)')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.02, bar.get_y() + bar.get_height()/2, f'R$ {w:.2f}M', va='center', fontsize=10)
plt.tight_layout()
plt.show()


### 4. Freight Ratio Analysis Across Categories
Investigating how freight burdens vary across product categories.


In [ ]:
top15 = cat_perf.head(15).copy()
top15['Freight_Ratio_%'] = (top15['Freight'] / top15['Revenue'] * 100).round(2)
top15 = top15.sort_values('Freight_Ratio_%', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=top15, x='Freight_Ratio_%', y='category', palette='Oranges_r', ax=ax)
ax.set_title('Freight Share of Total Order Value (Top 15 Categories)', fontsize=14, fontweight='bold')
ax.set_xlabel('Freight % of Total Revenue')
for i, v in enumerate(top15['Freight_Ratio_%']):
    ax.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()
